# Lekcja 3.1 — eksperymenty: gry i minimax

Uruchamiaj komórki po kolei. Przed każdym eksperymentem zapisz swoje przewidywanie w komórce „Twoje przewidywanie” (kliknij ją dwukrotnie, żeby edytować), a dopiero potem uruchom kod.

Pierwsza komórka wybiera implementację:

- `"mine"` używa Twojego `task.py`; najpierw rozwiąż część B zadania;
- `"reference"` używa implementacji wzorcowej kursu. Wpisz `IMPLEMENTATION = "reference"`, jeśli chcesz uruchomić notebook przed skończeniem zadania.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    import task as algorithms
else:
    from introai.games import reference as algorithms

Funkcja `run` uruchamia jedno przeszukiwanie i zwraca jego wynik, liczbę odwiedzonych pozycji i czas w sekundach. Pozycje liczymy po wywołaniach `game.result`, a pozycję startową też liczymy jako odwiedzoną. Funkcja `positions_by_ply` przechodzi całe drzewo gry i na każdym poziomie liczy węzły oraz różne pozycje. Minimax z tablicą transpozycji (`memoized_minimax_search`) pochodzi z wersji wzorcowej kursu.

In [ ]:
import time
from collections import Counter

import matplotlib.pyplot as plt

from introai.games import TIC_TAC_TOE, reference
from introai.games.players import play, random_player
from introai.testing.games import positions_visited
from introai.trace import TraceLimitExceeded, record

HINT = 'check task.py (or set IMPLEMENTATION = "reference" in the first cell)'


def run(search, game, state, limit=1_000_000):
    """Run one search; return its result, the positions visited and the time in seconds."""
    start = time.perf_counter()
    try:
        with record(limit=limit, kinds={"result"}) as trace:
            result = search(game, state)
    except NotImplementedError:
        raise RuntimeError(f"{search.__name__} is not written yet - {HINT}") from None
    except (TraceLimitExceeded, RecursionError):
        message = f"{search.__name__} visited more than {limit} positions - {HINT}"
        raise RuntimeError(message) from None
    return result, positions_visited(trace), time.perf_counter() - start


def positions_by_ply(game):
    """The plies, the nodes of the game tree and the distinct positions at every ply."""
    nodes, distinct = Counter(), {}
    stack = [(game.initial, 0)]
    while stack:
        state, ply = stack.pop()
        nodes[ply] += 1
        distinct.setdefault(ply, set()).add(state)
        if not game.is_terminal(state):
            stack.extend((game.result(state, a), ply + 1) for a in game.actions(state))
    plies = sorted(nodes)
    return plies, [nodes[p] for p in plies], [len(distinct[p]) for p in plies]

## Eksperyment 1: całe drzewo kółka i krzyżyka

Szukając pierwszego ruchu na pustej planszy, minimax przechodzi całe drzewo gry. Porównamy go z minimaksem, który pamięta pozycje już policzone. Obliczenia trwają kilka sekund.

**Twoje przewidywanie.** Ile pozycji odwiedzi Twój minimax? Ile z nich jest różnych? Ile razy mniej pracy wykona wersja z pamięcią?

*Moje przewidywanie:* …

In [ ]:
(value, move), visited, seconds = run(algorithms.minimax_search, TIC_TAC_TOE, TIC_TAC_TOE.initial)
_, memo_visited, memo_seconds = run(
    reference.memoized_minimax_search, TIC_TAC_TOE, TIC_TAC_TOE.initial
)
plies, nodes, distinct = positions_by_ply(TIC_TAC_TOE)
print(f"value of the empty board: {value}, first move: {move}")
print()
print(f"{'':<24}{'positions':>10}{'seconds':>9}")
print(f"{'minimax':<24}{visited:>10}{seconds:>9.2f}")
print(f"{'minimax with a table':<24}{memo_visited:>10}{memo_seconds:>9.2f}")
print(f"{'distinct positions':<24}{sum(distinct):>10}")
print()
repeats = sum(nodes) / sum(distinct)
print(f"the tree has {sum(nodes)} nodes: a position appears {repeats:.0f} times on average")

**Do przemyślenia.** Wersja z pamięcią też wywołuje `game.result` częściej, niż jest różnych pozycji. Dlaczego? W której lekcji modułu 2 pojawił się ten sam pomysł i jak się tam nazywał?

## Eksperyment 2: ile węzłów jest na każdym poziomie

Na każdym poziomie drzewa gry, od pustej planszy (poziom 0) do pełnej (poziom 9), liczymy węzły i różne pozycje. Oś pionowa ma skalę logarytmiczną.

**Twoje przewidywanie.** Na którym poziomie drzewo ma najwięcej węzłów? Dlaczego nie na ostatnim?

*Moje przewidywanie:* …

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
width = 0.4
ax.bar(
    [p - width / 2 for p in plies],
    nodes,
    width,
    color="#2a78d6",
    edgecolor="#fcfcfb",
    linewidth=1,
    label="nodes of the game tree",
)
ax.bar(
    [p + width / 2 for p in plies],
    distinct,
    width,
    color="#eb6834",
    edgecolor="#fcfcfb",
    linewidth=1,
    label="distinct positions",
)
peak = max(plies, key=lambda p: nodes[p])
ax.annotate(
    f"{nodes[peak]:,}",
    (peak - width / 2, nodes[peak]),
    textcoords="offset points",
    xytext=(0, 4),
    ha="center",
    color="#0b0b0b",
)
ax.set_yscale("log")
ax.set_xticks(plies)
ax.set_title(
    "Tic-tac-toe: nodes and distinct positions at each ply (log scale)",
    loc="left",
    color="#0b0b0b",
)
ax.set_xlabel("ply: moves made so far", color="#52514e")
ax.legend(frameon=False, loc="upper left", labelcolor="#0b0b0b")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
fig.tight_layout()

**Do przemyślenia.** Od piątego półruchu niektóre partie już się kończą. Jak to widać na wykresie? Na którym poziomie różnica między liczbą węzłów a liczbą różnych pozycji jest największa i skąd się bierze?

## Eksperyment 3: partie

Najpierw minimax gra sam ze sobą. Potem rozgrywa 100 partii z graczem, który wybiera ruchy losowo. W partiach o numerach parzystych minimax gra krzyżykami, a w nieparzystych kółkami. Minimax jest deterministyczny, więc gracz `remembering` zapamiętuje jego ruch w każdej pozycji i nie liczy go drugi raz.

**Twoje przewidywanie.** Jak skończy się partia minimaksu z minimaksem? Ile partii z graczem losowym minimax przegra? Czy wygra wszystkie?

*Moje przewidywanie:* …

In [ ]:
def remembering(search):
    """A player that plays the move of `search` and remembers it in every position."""
    moves = {}

    def player(game, state):
        if state not in moves:
            (_, moves[state]), _, _ = run(search, game, state)
        return moves[state]

    return player


perfect = remembering(algorithms.minimax_search)
duel = play(TIC_TAC_TOE, {"X": perfect, "O": perfect})
print(f"minimax against minimax: {duel.winner or 'draw'} after {len(duel.moves)} moves")
print(TIC_TAC_TOE.render(duel.state))

results = {"X": Counter(), "O": Counter()}
for number in range(100):
    side, other = ("X", "O") if number % 2 == 0 else ("O", "X")
    result = play(TIC_TAC_TOE, {side: perfect, other: random_player(number)})
    if result.winner is None:
        results[side]["draw"] += 1
    else:
        results[side]["win" if result.winner == side else "loss"] += 1
print()
print(f"{'minimax plays':<15}{'wins':>6}{'draws':>7}{'losses':>8}")
for side in ("X", "O"):
    r = results[side]
    print(f"{side:<15}{r['win']:>6}{r['draw']:>7}{r['loss']:>8}")

**Do przemyślenia.** Czy minimax przegrał choć jedną partię? Dlaczego kółkami wygrywa rzadziej niż krzyżykami? Czy wygrał każdą partię, którą dało się wygrać? Pomyśl, co minimax zakłada o przeciwniku.